# Hospital Daily Summary Gold Build

Builds `caresync_dbws.gold.hospital_daily_summary` from the STTM workbook `sttm_silver_to_gold.xlsx`. This notebook follows the documented grain of one row per hospital per calendar day and adds day-over-day trend metrics.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS caresync_dbws.gold;

CREATE OR REPLACE TABLE caresync_dbws.gold.hospital_daily_summary AS
WITH daily_base AS (
  SELECT
    lr.hospital_id,
    h.hospital_name,
    lr.test_date AS summary_date,
    c.day_name,
    c.is_weekend,
    CAST(COUNT(*) AS INT) AS total_tests,
    CAST(COUNT(DISTINCT lr.patient_id) AS INT) AS unique_patients_tested,
    CAST(COUNT(DISTINCT lr.doctor_id) AS INT) AS active_doctor_count,
    CAST(SUM(CASE WHEN lr.result_status = 'Abnormal' THEN 1 ELSE 0 END) AS INT) AS abnormal_test_count,
    CAST(SUM(CASE WHEN lr.result_status = 'Critical' THEN 1 ELSE 0 END) AS INT) AS critical_test_count
  FROM caresync_dbws.silver.lab_results lr
  LEFT JOIN caresync_dbws.silver.hospitals h
    ON lr.hospital_id = h.hospital_id
  LEFT JOIN caresync_dbws.silver.calender c
    ON c.datekey = CAST(date_format(lr.test_date, 'yyyyMMdd') AS INT)
  GROUP BY
    lr.hospital_id,
    h.hospital_name,
    lr.test_date,
    c.day_name,
    c.is_weekend
), daily_rates AS (
  SELECT
    hospital_id,
    hospital_name,
    summary_date,
    day_name,
    is_weekend,
    total_tests,
    unique_patients_tested,
    active_doctor_count,
    abnormal_test_count,
    critical_test_count,
    CAST(ROUND(CASE WHEN total_tests = 0 THEN NULL ELSE (abnormal_test_count * 100.0) / total_tests END, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(CASE WHEN total_tests = 0 THEN NULL ELSE (critical_test_count * 100.0) / total_tests END, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM daily_base
), daily_trends AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily_rates
)
SELECT
  hospital_id,
  hospital_name,
  summary_date,
  day_name,
  is_weekend,
  total_tests,
  unique_patients_tested,
  active_doctor_count,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  CAST(prev_day_total_tests AS INT) AS prev_day_total_tests,
  CAST(
    ROUND(
      CASE
        WHEN prev_day_total_tests IS NULL OR prev_day_total_tests = 0 THEN NULL
        ELSE ((total_tests - prev_day_total_tests) * 100.0) / prev_day_total_tests
      END,
      2
    ) AS DECIMAL(6,2)
  ) AS test_volume_change_pct
FROM daily_trends;


In [0]:
updated_df = spark.read.table("caresync_dbws.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))
